<a id='top'></a>

# Webscraping of TransferMarkt Data
##### Notebook to scrape raw data from [TransferMarkt](https://www.transfermarkt.co.uk/) using [Beautifulsoup](https://pypi.org/project/beautifulsoup4/).

### By [Edd Webster](https://www.twitter.com/eddwebster)
Notebook first written: 13/09/2020<br>
Notebook last updated: 04/12/2020

![title](../../img/transfermarkt-logo-banner.png)

Click [here](#section5) to jump straight to the Exploratory Data Analysis section and skip the [Task Brief](#section2), [Data Sources](#section3), and [Data Engineering](#section4) sections. Or click [here](#section6) to jump straight to the Conclusion.

___

<a id='sectionintro'></a>

## <a id='import_libraries'>Introduction</a>
This notebook scrapes data from [TransferMarkt](https://www.transfermarkt.co.uk/) using [Beautifulsoup](https://pypi.org/project/beautifulsoup4/) and the [Tyrone Mings web scraper](https://github.com/FCrSTATS/tyrone_mings) by [FCrSTATS](https://twitter.com/FC_rstats). This landed data is then manipulated as DataFrames using [pandas](http://pandas.pydata.org/).

For more information about this notebook and the author, I'm available through all the following channels:
*    [eddwebster.com](https://www.eddwebster.com/);
*    edd.j.webster@gmail.com;
*    [@eddwebster](https://www.twitter.com/eddwebster);
*    [linkedin.com/in/eddwebster](https://www.linkedin.com/in/eddwebster/);
*    [github/eddwebster](https://github.com/eddwebster/);
*    [public.tableau.com/profile/edd.webster](https://public.tableau.com/profile/edd.webster);
*    [kaggle.com/eddwebster](https://www.kaggle.com/eddwebster); and
*    [hackerrank.com/eddwebster](https://www.hackerrank.com/eddwebster).

![title](../../img/fifa21eddwebsterbanner.png)

The accompanying GitHub repository for this notebook can be found [here](https://github.com/eddwebster/football_analytics) and a static version of this notebook can be found [here](https://nbviewer.jupyter.org/github/eddwebster/football_analytics/blob/master/notebooks/A%29%20Web%20Scraping/TransferMarkt%20Web%20Scraping%20and%20Parsing.ipynb).

___

<a id='sectioncontents'></a>

## <a id='notebook_contents'>Notebook Contents</a>
1.    [Notebook Dependencies](#section1)<br>
2.    [Project Brief](#section2)<br>
3.    [Data Sources](#section3)<br>
      1.    [Introduction](#section3.1)<br>
      2.    [Data Dictionary](#section3.2)<br>
      3.    [Creating the DataFrame](#section3.3)<br>
      4.    [Initial Data Handling](#section3.4)<br>
      5.    [Export the Raw DataFrame](#section3.5)<br>         
4.    [Data Engineering](#section4)<br>
      1.    [Introduction](#section4.1)<br>
      2.    [Columns of Interest](#section4.2)<br>
      3.    [String Cleaning](#section4.3)<br>
      4.    [Converting Data Types](#section4.4)<br>
      5.    [Export the Engineered DataFrame](#section4.5)<br>
5.    [Exploratory Data Analysis (EDA)](#section5)<br>
      1.    [...](#section5.1)<br>
      2.    [...](#section5.2)<br>
      3.    [...](#section5.3)<br>
6.    [Summary](#section6)<br>
7.    [Next Steps](#section7)<br>
8.    [Bibliography](#section8)<br>

___

<a id='section1'></a>

## <a id='#section1'>1. Notebook Dependencies</a>

This notebook was written using [Python 3](https://docs.python.org/3.7/) and requires the following libraries:
*    [`Jupyter notebooks`](https://jupyter.org/) for this notebook environment with which this project is presented;
*    [`NumPy`](http://www.numpy.org/) for multidimensional array computing;
*    [`pandas`](http://pandas.pydata.org/) for data analysis and manipulation;
*    [`Beautifulsoup`](https://pypi.org/project/beautifulsoup4/) for web scraping; and
*    [`matplotlib`](https://matplotlib.org/contents.html?v=20200411155018) for data visualisations;

All packages used for this notebook except for BeautifulSoup can be obtained by downloading and installing the [Conda](https://anaconda.org/anaconda/conda) distribution, available on all platforms (Windows, Linux and Mac OSX). Step-by-step guides on how to install Anaconda can be found for Windows [here](https://medium.com/@GalarnykMichael/install-python-on-windows-anaconda-c63c7c3d1444) and Mac [here](https://medium.com/@GalarnykMichael/install-python-on-mac-anaconda-ccd9f2014072), as well as in the Anaconda documentation itself [here](https://docs.anaconda.com/anaconda/install/).

### Import Libraries and Modules

In [1]:
# Python ≥3.5 (ideally)
import platform
import sys, getopt
assert sys.version_info >= (3, 5)
import csv

# Import Dependencies
%matplotlib inline

# Math Operations
import numpy as np
from math import pi

# Datetime
import datetime
from datetime import date
import time

# Data Preprocessing
import pandas as pd
import os
import re
import random
from io import BytesIO
from pathlib import Path

# Reading directories
import glob
import os
from os.path import basename

# Flatten lists
from functools import reduce

# Working with JSON
import json
from pandas.io.json import json_normalize

# Web Scraping
import requests
from bs4 import BeautifulSoup
import re

# APIs
from tyrone_mings import * 

# Fuzzy Matching - Record Linkage
import recordlinkage
import jellyfish
import numexpr as ne

# Data Visualisation
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-whitegrid')
import missingno as msno

# Progress Bar
from tqdm import tqdm

# Display in Jupyter
from IPython.display import Image, YouTubeVideo
from IPython.core.display import HTML

# Ignore Warnings
import warnings
warnings.filterwarnings(action="ignore", message="^internal gelsd")

print('Setup Complete')

Setup Complete


In [2]:
# Python / module versions used here for reference
print('Python: {}'.format(platform.python_version()))
print('NumPy: {}'.format(np.__version__))
print('pandas: {}'.format(pd.__version__))
print('matplotlib: {}'.format(mpl.__version__))
print('Seaborn: {}'.format(sns.__version__))

Python: 3.7.6
NumPy: 1.18.1
pandas: 1.0.1
matplotlib: 3.1.3
Seaborn: 0.10.0


### Defined Variables

In [3]:
# Define today's date
today = datetime.datetime.now().strftime('%d/%m/%Y').replace('/', '')

### Defined Filepaths

In [4]:
# Set up initial paths to subfolders
base_dir = os.path.join('..', '..', )
data_dir = os.path.join(base_dir, 'data')
data_dir_tm = os.path.join(base_dir, 'data', 'tm')
img_dir = os.path.join(base_dir, 'img')
fig_dir = os.path.join(base_dir, 'img', 'fig')
video_dir = os.path.join(base_dir, 'video')

### Notebook Settings

In [5]:
pd.set_option('display.max_columns', None)

---

<a id='section2'></a>

## <a id='#section2'>2. Project Brief</a>
This notebook scrapes data from [TransferMarkt](https://www.transfermarkt.co.uk/) using [Beautifulsoup](https://pypi.org/project/beautifulsoup4/) and the [Tyrone Mings web scraper](https://github.com/FCrSTATS/tyrone_mings) by [FCrSTATS](https://twitter.com/FC_rstats). This landed data is then manipulated as DataFrames using [pandas](http://pandas.pydata.org/).

The data of player values produced in this notebook is exported to CSV. This data can be further analysed in Python, joined to other datasets, or explored using dashboarding tools such as Tableau or PowerBI, or explores in a spreadsheet such as Microsoft Excel or Google Sheets.

---

<a id='section3'></a>

## <a id='#section3'>3. Data Sources</a>

### <a id='#section3.1'>3.1. Introduction</a>
[TransferMarkt](https://www.transfermarkt.co.uk/) is a German-based website owned by [Axel Springer](https://www.axelspringer.com/en/) and is the leading website for the football transfer market. The website posts football related data, including: scores and results, football news, transfer rumours, and most usefully for us - calculated estimates ofthe market values for teams and individual players.

To read more about how these estimations are made, [Beyond crowd judgments: Data-driven estimation of market value in association football](https://www.sciencedirect.com/science/article/pii/S0377221717304332) by Oliver Müllera, Alexander Simons, and Markus Weinmann does an excellent job of explaining how the estimations are made and their level of accuracy.

Before conducting our EDA, the data needs to be imported as a DataFrame in the Data Sources section [Section 3](#section3) and cleaned in the Data Engineering section [Section 4](#section4).

We'll be using the [pandas](http://pandas.pydata.org/) library to import our data to this workbook as a DataFrame.

### <a id='#section3.2'>3.2. Data Dictionaries</a>
The [TransferMarkt](https://www.transfermarkt.co.uk/) dataset has six features (columns) with the following definitions and data types:

| Feature     | Data type    |
|------|-----|
| `position_number`    | object     |
| `position_description`    | object     |
| `name`    | object     |
| `dob`    | object     |
| `nationality`    | object     |
| `value`    | object     |

### <a id='#section3.3'>3.3. Creating the DataFrame - scraping the data</a>

#### <a id='#section3.3.1.'>3.3.1. League Codes</a>
Before scraping data from [TransferMarkt](https://www.transfermarkt.co.uk/), we need to look at the leagues in England that we wish to scrape.

The [Tyrone Mings web scraper](https://github.com/FCrSTATS/tyrone_mings) by [FCrSTATS](https://github.com/FCrSTATS) for [TransferMarkt](https://www.transfermarkt.co.uk/) is made up of two parts:
1.    In the first part, the scraper takes the webpages for each of the individual leagues  e.g. the Championship, and extract the hyperlinks to the pages of all the individual teams in the league table.
2.    In the second part the script, the webscraper uses the list of invidual teams hyperlinks collected in part 1 to then collect the hyperlinks for each of the players for those teams. From this, the scraper can then extract the information we need for each of these players. This information is downloaded in two parts - bio and status. This is then joined together using the player_id.

This information collected for all the players is converted to a [pandas](http://pandas.pydata.org/) DataFrame from which we can view and manipulate the data.

An example webpage for a football league is the following: https://www.transfermarkt.co.uk/jumplist/startseite/wettbewerb/GB1/plus/?saison_id=2019. As we can see, between the subdirectory path of `'/wettbewerb/'` and the `'/plus/'`, there is a 3 or 4 digit code. For The Premier League, the code is GB1.

In order to scrape the webpages from [TransferMarkt](https://www.transfermarkt.co.uk/), the codes the leagues we require, need to be recorded from [TransferMarkt](https://www.transfermarkt.co.uk/). The following leagues are all the leagues that feature on the latest FBref 'Big 5' European leagues dataset, the EFL dataset, and the the [FIFA 21 player dataset](https://www.kaggle.com/stefanoleone992/fifa-21-complete-player-dataset), all of which are datasets for which this [TransferMarkt](https://www.transfermarkt.co.uk/) data will be subsequently matched to.

| League Name on FIFA    | Country    | Corresponding [TransferMarkt](https://www.transfermarkt.co.uk/) League Code    |
|------|-----|-----|
| Österreichische Fußball-Bundesliga    | Austria    | A1    |
| SAF    | Argentina    | AR1N    |
| Hyundai A-League    | Australia    | AUS1    |
| Belgium Pro League    | Belgium    | BE1    |
| Raiffeisen Super League    | Switzerland    | C1    |
| Campeonato Scotiabank    | Chile    | CLPD    |
| Liga Dimayor    | Colombia    | COL1    |
| CSL    | China    | CSL    |
| Superliga    | Denmark    | DK1    |
| LaLiga Santander    | Spain    | ES1    |
| LaLiga 1 I 2 I 3    | Spain    | ES2    |
| Finnliiga    | Finland    | FI1    |
| Ligue 1 Conforama    | France    | FR1    |
| Domino’s Ligue 2    | France    | FR2    |
| Premier League    | England    | GB1    |
| EFL Championship    | England    | GB2    |
| EFL League One    | England    | GB3    |
| EFL League Two    | England    | GB4    |
| Hellas Liga    | Greece    | GR1    |
| SSE Airtricity League    | Ireland    | IR1    |
| Serie A TIM    | Italy    | IT1    |
| Calcio B    | Italy    | IT2    |
| Meiji Yasuda J1 League    | Japan    | JAP1    |
| Croatia Liga    | Croatia    | KR1    |
| Bundesliga    | Germany    | L1    |
| Bundesliga 2    | Germany    | L2    |
| 3. Liga    | Germany    | L3    |
| LIGA Bancomer MX    | Mexico    | MEX1    |
| Major League Soccer    | United States    | MLS1    |
| Eredivisie    | Netherlands    | NL1    |
| Eliteserien    | Norway    | NO1    |
| Ekstraklasa    | Poland    | PL1    |
| Liga NOS    | Portugal    | PO1    |
| Romania Liga I    | Romania    | RO1    |
| K LEAGUE Classic    | South Korea    | RSK1    |
| League of Russia    | Russia    | RU1    |
| Saudi Professional    | League	Saudi Arabia    | SA1    |
| Scottish Premiership    | Scotland    | SC1    |
| Allsvenskan    | Sweden    | SE1    |
| South African FL    | South Africa    | SFA1    |
| Süper Lig    | Turkey    | TR1    |
| Česká Liga    | Czech Republic    | TS1    |
| UAE Gulf League    | United Arab Emirates    | UAE1    |
| Ukraine Liga    | Ukraine    | UKR1    |


Unfortunately, on writing this notebook, the following leagues are that is present on FIFA but cannot scraped from [TransferMarkt](https://www.transfermarkt.co.uk/) with the script and will and therefore not be included in the dataset.

| League Name on FIFA    | Country    | Corresponding [TransferMarkt](https://www.transfermarkt.co.uk/) League Code    |
|------|-----|-----|
| League of Russia    | Russia    | RU1    |

#### <a id='#section3.3.2.'>3.3.2. Define Season and Leagues to Scrape</a>

In [6]:
# Define variables and lists

## Define season
season = '2020'    # '2020' for the 20/21 season

## Define list of leagues for players to scrape
lst_leagues = ['A1',
               'AR1N',
               'AUS1',
               'BE1',
               'C1',
               'CLPD',
               'COL1',
               'CSL',
               'DK1',
               'ES1',
               'ES2',
               'FI1',
               'FR1',
               'FR2',
               'GB1',
               'GB2',
               'GB3',
               'GB4',
               'GR1',
               'IR1',
               'IT1',
               'IT2',
               'JAP1',
               'KR1',
               'L1',
               'L2',
               'L3',
               'MEX1',
               'MLS1',
               'NL1',
               'NO1',
               'PL1',
               'PO1',
               'RO1',
               'RSK1',
               'RU1',
               'SA1',
               'SC1',
               'SE1',
               'SFA1',
               'TR1',
               'TS1',
               'UAE1',
               'UKR1'
              ]

In [7]:
# Create 'Full Season' and 'Short Season' strings

## Full season
full_season_string = str(int(season)) + '/' + str(int(season) + 1)

## Short season
short_season_string = str((str(int(season))[-2:]) + (str(int(season) + 1)[-2:]))

In [8]:
full_season_string

'2020/2021'

In [9]:
short_season_string

'2021'

#### <a id='#section3.3.3.'>3.3.3. Player URLs</a>

In [10]:
# Run this script to download all the URLs for the players of interest from TransferMarkt

## Start timer
tic = datetime.datetime.now()

## Create empty list
lst_player_urls = []

## Print time scraping started
print(f'Scraping started at: {tic}')

## Scrape information for each player
for league in lst_leagues:
    try:
        lst_output = get_player_urls_from_league_page(f'https://www.transfermarkt.co.uk/championship/startseite/wettbewerb/{league}/plus/?saison_id={season}', verbose=True)
        lst_player_urls.append(lst_output)
        print(f'All player URLs for the {league} league appended.')
    except:
        pass

## End timer
toc = datetime.datetime.now()

## Print time scraping ended
print(f'Scraping ended at: {toc}')

## Flatten nested list into a single list
lst_player_urls = reduce(lambda x,y: x+y, lst_player_urls)

## No. URLs i.e. players
len_player_urls = len(lst_player_urls)

## Calculate time take
total_time = (toc-tic).total_seconds()
print(f'Time taken to scrape the {len_player_urls:,} player urls for the {full_season_string} season is: {total_time/60:0.2f} minutes.')

Scraping started at: 2020-12-07 14:50:35.925099
fk austria wien players added
fc admira wacker modling players added
red bull salzburg players added
wolfsberger ac players added
skn st polten players added
sk rapid wien players added
sk sturm graz players added
sc rheindorf altach players added
wsg wattens players added
lask linz players added
tsv hartberg players added
sv ried players added
All player URLs for the A1 league appended.
club atletico san lorenzo de almagro players added
club atletico banfield players added
club atletico central cordoba sde  players added
club deportivo godoy cruz players added
ca independiente de avellaneda players added
club atletico aldosivi mdp  players added
club atletico colon santa fe  players added
club atletico lanus players added
club atletico huracan players added
estudiantes de la plata players added
club atletico boca juniors players added
club atletico tucuman players added
club atletico river plate players added
club atletico newells old bo

wolverhampton wanderers players added
aston villa players added
manchester united players added
west bromwich albion players added
fc arsenal players added
tottenham hotspur players added
All player URLs for the GB1 league appended.
fc middlesbrough players added
swansea city players added
afc bournemouth players added
stoke city players added
sheffield wednesday players added
fc barnsley players added
derby county players added
wycombe wanderers players added
fc reading players added
cardiff city players added
queens park rangers players added
birmingham city players added
luton town players added
huddersfield town players added
blackburn rovers players added
fc millwall players added
nottingham forest players added
fc watford players added
fc brentford players added
coventry city players added
rotherham united players added
bristol city players added
norwich city players added
preston north end players added
All player URLs for the GB2 league appended.
charlton athletic players added

new england revolution players added
inter miami cf players added
d c united players added
minnesota united fc players added
atlanta united fc players added
fc cincinnati players added
All player URLs for the MLS1 league appended.
vvv venlo players added
fc twente enschede players added
az alkmaar players added
fc emmen players added
psv eindhoven players added
fortuna sittard players added
ajax amsterdam players added
vitesse arnheim players added
ado den haag players added
fc groningen players added
heracles almelo players added
rkc waalwijk players added
willem ii tilburg players added
sparta rotterdam players added
feyenoord rotterdam players added
fc utrecht players added
pec zwolle players added
sc heerenveen players added
All player URLs for the NL1 league appended.
stabaek if players added
kristiansund bk players added
sandefjord fotball players added
ik start kristiansand players added
molde fk players added
mjondalen if players added
brann bergen players added
valerenga oslo 

In [11]:
# List length i.e. total players
len(lst_player_urls)

21940

In [ ]:
# Write some code here that saves these URLs to a one column DataFrame

#### <a id='#section3.3.3.'>3.3.3. Bio Information</a>

In [48]:
# Run this script to scrape latest version of bio data from TransferMarkt

## Start timer
tic = datetime.datetime.now()

## Create empty list
lst_df_bio = []

## Print time scraping started
print(f'Scraping started at: {tic}')

## Scrape bio information for each player
for player_page in lst_player_urls:
    try:
        dict_output = tm_pull(player_page, player_bio=True, output='pandas')
        df = dict_output['player_bio']
        lst_df_bio.append(df)
        print(f'Bio data appended for: {player_page}.')
    except:
        pass

## Concatenate DataFrames
df_bio = pd.concat(lst_df_bio)

## Print time scraping ended
print(f'Scraping ended at: {toc}')

## Create attribute for the season    
df_bio['season'] = full_season_string

## End timer
toc = datetime.datetime.now()

## Calculate time take
total_time = (toc-tic).total_seconds()
print(f'Time taken to scrape the bio data of the {len_player_urls:,} players for the {full_season_string} season is: {total_time/60:0.2f} minutes.')

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHA

Time taken to scrape the bio data of the 21965 players for the 20/21 season is: 295.74 minutes.


In [64]:
# Display DataFrame
df_bio.head()

,player_id,player_name,day_of_birth,month_of_birth,year_of_birth,pob,cob,dob,position,height,foot,second_citizenship,season
0,206906,dominik baumgartner,20,7,1996,Horn,Austria,1996-07-20,None,187,right,None,2020-2021
0,336382,dejan joveljic,7,8,1999,Bijeljina,Bosnia-Herzegovina,1999-08-07,None,182,right,Bosnia-Herzegovina,2020-2021
0,52821,mario leitgeb,30,6,1988,Graz,Austria,1988-06-30,None,183,right,None,2020-2021
0,444018,eliel peretz,18,11,1996,Bat Yam,Israel,1996-11-18,None,189,right,None,2020-2021
0,145010,marc andre schmerböck,1,4,1994,Feldbach,Austria,1994-04-01,None,180,left,None,2020-2021


In [63]:
df_bio.shape

(21965, 13)

##### Export DataFrame

In [67]:
# Export DataFrame as a CSV file

## Export a copy to the 'archive' subfolder of the TM folder, including the date
df_bio.to_csv(data_dir_tm + f'/raw/{short_season_string}/bio/archive/' + f'tm_player_bio_all_{short_season_string}_last_updated_{today}.csv', index=None, header=True)

## Export another copy to the TM folder called 'latest' (can be overwritten)
df_bio.to_csv(data_dir_tm + f'/raw/{short_season_string}/bio/' + f'tm_player_bio_all_{short_season_string}_latest.csv', index=None, header=True)

#### <a id='#section3.3.4.'>3.3.4. Status Information</a>

In [53]:
# Run this script to scrape latest version of status data from TransferMarkt

## Start timer
tic = datetime.datetime.now()

## Create empty list
lst_df_status = []

## Print time scraping started
print(f'Scraping started at: {tic}')

## Scrape status information for each player
for player_page in lst_player_urls:
    try:
        dict_output = tm_pull(player_page, player_status=True, output='pandas')
        df = dict_output['player_status']
        lst_df_status.append(df)
        print(f'Status data appended for: {player_page}.')
    except:
        pass

## Concatenate DataFrames
df_status = pd.concat(lst_df_status)

## Print time scraping ended
print(f'Scraping ended at: {toc}')

## Create attribute for the season    
df_status['season'] = full_season_string

## End timer
toc = datetime.datetime.now()

## Calculate time take
total_time = (toc-tic).total_seconds()
print(f'Time taken to scrape the status data of the {len_player_urls:,} players for the {full_season_string} season is: {total_time/60:0.2f} minutes.')

Scraping started at: 2020-12-05 04:44:09.834141


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHA

Scraping ended at: 2020-12-05 04:33:32.993690
Time taken to scrape the status data of the 21965 players for the 20/21 season is: 311.60 minutes.


In [65]:
# Display DataFrame
df_status.head()

,player_id,current_club,current_club_country,market_value,joined,contract_expires,contract_option,on_loan_from,on_loan_from_country,loan_contract_expiry,player_agent,season
0,531217,wolfsberger ac,austria,200000,2020-06-01,None,None,None,None,None,Selection Fussballconsulting,2020-2021
0,451317,sv ried,austria,400000,2019-07-01,None,None,None,None,None,More than Sport GmbH,2020-2021
0,445038,sv ried,austria,75000,2020-09-21,None,None,None,None,None,ONE ON ONE Sports Management,2020-2021
0,159581,sv ried,austria,200000,2019-07-01,None,None,None,None,None,Relatives,2020-2021
0,394070,sv ried,austria,75000,2019-07-01,None,None,None,None,None,None,2020-2021


In [66]:
df_status.shape

(3581, 12)

##### Export DataFrame

In [54]:
# Export DataFrame as a CSV file

## Export a copy to the 'archive' subfolder of the TM folder, including the date
df_status.to_csv(data_dir_tm + f'/raw/{short_season_string}/status/archive/' + f'tm_player_status_all_{short_season_string}_last_updated_{today}.csv', index=None, header=True)

## Export another copy to the TM folder called 'latest' (can be overwritten)
df_status.to_csv(data_dir_tm + f'/raw/{short_season_string}/status/' + f'tm_player_status_all_{short_season_string}_latest.csv', index=None, header=True)

#### <a id='#section3.3.5.'>3.3.5. Transfer History</a>

In [29]:
# Run this script to scrape latest version of transfer history data from TransferMarkt

## Start timer
tic = datetime.datetime.now()

## Create empty list
lst_df_th = []

## Print time scraping started
print(f'Scraping started at: {tic}')

## Scrape bio information for each player
for player_page in lst_player_urls:
    try:
        dict_output = tm_pull(player_page, transfer_history=True, output='pandas')
        df = dict_output['transfer_history']
        lst_df_th.append(df)
        print(f'Transfer history data appended for: {player_page}.')
    except:
        pass

## Concatenate DataFrames
df_th = pd.concat(lst_df_th)

## Print time scraping ended
print(f'Scraping ended at: {toc}')

## Create attribute for the season    
df_th['season'] = full_season_string

## End timer
toc = datetime.datetime.now()

## Calculate time take
total_time = (toc-tic).total_seconds()
print(f'Time taken to scrape the transfer history data of the {len_player_urls:,} players for the {full_season_string} season is: {total_time/60:0.2f} minutes.')

Scraping started at: 2020-12-05 16:19:57.492112
Transfer history data appended for: https://www.transfermarkt.com/samuel-oum-gouet/profil/spieler/452238.
Transfer history data appended for: https://www.transfermarkt.com/emanuel-schreiner/profil/spieler/59132.
Transfer history data appended for: https://www.transfermarkt.com/nosa-iyobosa-edokpolor/profil/spieler/276841.
Transfer history data appended for: https://www.transfermarkt.com/jakob-odehnal/profil/spieler/526353.
Transfer history data appended for: https://www.transfermarkt.com/tino-casali/profil/spieler/169271.
Transfer history data appended for: https://www.transfermarkt.com/lars-nussbaumer/profil/spieler/407503.
Transfer history data appended for: https://www.transfermarkt.com/alain-wiss/profil/spieler/58165.
Transfer history data appended for: https://www.transfermarkt.com/nana-kofi-babil/profil/spieler/734819.
Transfer history data appended for: https://www.transfermarkt.com/manfred-fischer/profil/spieler/269093.
Transfer h

Transfer history data appended for: https://www.transfermarkt.com/srdjan-grahovac/profil/spieler/123158.
Transfer history data appended for: https://www.transfermarkt.com/marcel-ritzmaier/profil/spieler/84162.
Transfer history data appended for: https://www.transfermarkt.com/adrian-hajdari/profil/spieler/388304.
Transfer history data appended for: https://www.transfermarkt.com/filip-stojkovic/profil/spieler/139751.
Transfer history data appended for: https://www.transfermarkt.com/thorsten-schick/profil/spieler/55574.
Transfer history data appended for: https://www.transfermarkt.com/christopher-dibon/profil/spieler/54607.
Transfer history data appended for: https://www.transfermarkt.com/dalibor-velimirovic/profil/spieler/437891.
Transfer history data appended for: https://www.transfermarkt.com/mateo-barac/profil/spieler/329629.
Transfer history data appended for: https://www.transfermarkt.com/taxiarchis-fountas/profil/spieler/192409.
Transfer history data appended for: https://www.trans

Transfer history data appended for: https://www.transfermarkt.com/philipp-sturm/profil/spieler/307696.
Transfer history data appended for: https://www.transfermarkt.com/sascha-horvath/profil/spieler/186365.
Transfer history data appended for: https://www.transfermarkt.com/dario-tadic/profil/spieler/50860.
Transfer history data appended for: https://www.transfermarkt.com/lukas-fadinger/profil/spieler/394018.
Transfer history data appended for: https://www.transfermarkt.com/jurgen-heil/profil/spieler/271223.
Transfer history data appended for: https://www.transfermarkt.com/abdoul-yoda/profil/spieler/811856.
Transfer history data appended for: https://www.transfermarkt.com/raphael-sallinger/profil/spieler/225360.
Transfer history data appended for: https://www.transfermarkt.com/thomas-rotter/profil/spieler/119231.
Transfer history data appended for: https://www.transfermarkt.com/seifedin-chabbi/profil/spieler/106862.
Transfer history data appended for: https://www.transfermarkt.com/filip-

Transfer history data appended for: https://www.transfermarkt.com/ivan-ljubic/profil/spieler/250028.
Transfer history data appended for: https://www.transfermarkt.com/tobias-koch/profil/spieler/434103.
Transfer history data appended for: https://www.transfermarkt.com/sandro-schendl/profil/spieler/527596.
Transfer history data appended for: https://www.transfermarkt.com/sandro-ingolitsch/profil/spieler/286627.
Transfer history data appended for: https://www.transfermarkt.com/tobias-schutzenauer/profil/spieler/243237.
Transfer history data appended for: https://www.transfermarkt.com/jakob-jantscher/profil/spieler/52269.
Transfer history data appended for: https://www.transfermarkt.com/andreas-kuen/profil/spieler/186798.
Transfer history data appended for: https://www.transfermarkt.com/jon-gorenc-stankovic/profil/spieler/245034.
Transfer history data appended for: https://www.transfermarkt.com/otar-kiteishvili/profil/spieler/352596.
Transfer history data appended for: https://www.transfer

Transfer history data appended for: https://www.transfermarkt.com/thomas-turner/profil/spieler/316729.
Transfer history data appended for: https://www.transfermarkt.com/petar-filipovic/profil/spieler/85922.
Transfer history data appended for: https://www.transfermarkt.com/mads-emil-madsen/profil/spieler/422950.
Transfer history data appended for: https://www.transfermarkt.com/dominik-reiter/profil/spieler/307667.
Transfer history data appended for: https://www.transfermarkt.com/valentino-muller/profil/spieler/307817.
Transfer history data appended for: https://www.transfermarkt.com/marvin-potzmann/profil/spieler/106869.
Transfer history data appended for: https://www.transfermarkt.com/mamoudou-karamoko/profil/spieler/646782.
Transfer history data appended for: https://www.transfermarkt.com/philipp-wiesinger/profil/spieler/159139.
Transfer history data appended for: https://www.transfermarkt.com/rene-renner/profil/spieler/212333.
Transfer history data appended for: https://www.transferm

...

Transfer history data appended for: https://www.transfermarkt.com/valeriy-yurchuk/profil/spieler/292396.
Transfer history data appended for: https://www.transfermarkt.com/oleksandr-pikhalyonok/profil/spieler/294780.
Transfer history data appended for: https://www.transfermarkt.com/oleksiy-chychykov/profil/spieler/43404.
Transfer history data appended for: https://www.transfermarkt.com/oleksandr-svatok/profil/spieler/244271.
Transfer history data appended for: https://www.transfermarkt.com/lucas-taylor/profil/spieler/387928.
Transfer history data appended for: https://www.transfermarkt.com/bogdan-sarnavskyi/profil/spieler/177276.
Transfer history data appended for: https://www.transfermarkt.com/dmytro-korkishko/profil/spieler/91380.
Transfer history data appended for: https://www.transfermarkt.com/douglas/profil/spieler/101304.
Transfer history data appended for: https://www.transfermarkt.com/yegor-yarmolyuk/profil/spieler/717411.
Transfer history data appended for: https://www.transfer

Transfer history data appended for: https://www.transfermarkt.com/volodymyr-yakimets/profil/spieler/396402.
Transfer history data appended for: https://www.transfermarkt.com/mihkel-ainsalu/profil/spieler/198967.
Transfer history data appended for: https://www.transfermarkt.com/ernest/profil/spieler/359678.
Transfer history data appended for: https://www.transfermarkt.com/maks-juraj-celic/profil/spieler/348768.
Transfer history data appended for: https://www.transfermarkt.com/yegor-klymenchuk/profil/spieler/375955.
Transfer history data appended for: https://www.transfermarkt.com/oleksandr-savoshko/profil/spieler/775824.
Transfer history data appended for: https://www.transfermarkt.com/china/profil/spieler/497719.
Transfer history data appended for: https://www.transfermarkt.com/oleksandr-romanchuk/profil/spieler/553772.
Transfer history data appended for: https://www.transfermarkt.com/welves/profil/spieler/654311.
Transfer history data appended for: https://www.transfermarkt.com/rafael

Transfer history data appended for: https://www.transfermarkt.com/oleksandr-kucherenko/profil/spieler/358956.
Transfer history data appended for: https://www.transfermarkt.com/artem-shchedryi/profil/spieler/123279.
Transfer history data appended for: https://www.transfermarkt.com/andriy-semenko/profil/spieler/379977.
Transfer history data appended for: https://www.transfermarkt.com/dmytro-fateev/profil/spieler/271635.
Transfer history data appended for: https://www.transfermarkt.com/artem-malysh/profil/spieler/461653.
Transfer history data appended for: https://www.transfermarkt.com/orest-lebedenko/profil/spieler/524964.
Transfer history data appended for: https://www.transfermarkt.com/yevgeniy-pasich/profil/spieler/315237.
Transfer history data appended for: https://www.transfermarkt.com/ivan-zotko/profil/spieler/305423.
Transfer history data appended for: https://www.transfermarkt.com/igor-snurnitsyn/profil/spieler/419119.
Transfer history data appended for: https://www.transfermarkt

Transfer history data appended for: https://www.transfermarkt.com/nazariy-rusyn/profil/spieler/366729.
Transfer history data appended for: https://www.transfermarkt.com/volodymyr-kostevych/profil/spieler/208469.
Transfer history data appended for: https://www.transfermarkt.com/bogdan-lednev/profil/spieler/423606.
Transfer history data appended for: https://www.transfermarkt.com/maksim-grechkin/profil/spieler/347270.
Transfer history data appended for: https://www.transfermarkt.com/lovro-cvek/profil/spieler/317467.
Transfer history data appended for: https://www.transfermarkt.com/tymofiy-sukhar/profil/spieler/404844.
Transfer history data appended for: https://www.transfermarkt.com/oleksandr-gladkyi/profil/spieler/53270.
Transfer history data appended for: https://www.transfermarkt.com/vitaliy-vernydub/profil/spieler/59973.
Transfer history data appended for: https://www.transfermarkt.com/joel-abu-hanna/profil/spieler/267330.
Transfer history data appended for: https://www.transfermarkt

Transfer history data appended for: https://www.transfermarkt.com/erik-gliha/profil/spieler/284608.
Transfer history data appended for: https://www.transfermarkt.com/rostyslav-rusyn/profil/spieler/289487.
Transfer history data appended for: https://www.transfermarkt.com/bogdan-boychuk/profil/spieler/337198.
Transfer history data appended for: https://www.transfermarkt.com/oleg-veremiyenko/profil/spieler/497222.
Transfer history data appended for: https://www.transfermarkt.com/maryan-mysyk/profil/spieler/316945.
Transfer history data appended for: https://www.transfermarkt.com/ostap-prytula/profil/spieler/620368.
Transfer history data appended for: https://www.transfermarkt.com/yuriy-kopyna/profil/spieler/432658.
Transfer history data appended for: https://www.transfermarkt.com/maksym-bilyi/profil/spieler/107803.
Transfer history data appended for: https://www.transfermarkt.com/mykola-kukharevych/profil/spieler/730126.
Transfer history data appended for: https://www.transfermarkt.com/iv

In [33]:
# Display DataFrame
df_th.head()

,transfered_from,transferred_to,market_values,transfer_fees,transfer_dates,transfer_season,country_to,country_from,transfer_types,internal_external_transfer,age_at_transfer,player_id,all_youth_clubs,season
0,apejes fc de mfou,sc rheindorf altach,0,0,2017-08-31,17/18,austria,cameroon,transfer,external,19,452238,apejes fc de mfou,2020/2021
0,sv ried,sc rheindorf altach,0,0,2013-07-01,13/14,austria,austria,transfer,external,24,59132,"aka linz,fussballakademie linz,usv st. ulrich,...",2020/2021
1,lask linz,sv ried,0,0,2011-07-01,11/12,austria,austria,transfer,external,22,59132,"aka linz,fussballakademie linz,usv st. ulrich,...",2020/2021
2,sc austria lustenau,lask linz,0,0,2010-06-30,09/10,austria,austria,loan,external,21,59132,"aka linz,fussballakademie linz,usv st. ulrich,...",2020/2021
3,lask linz,sc austria lustenau,0,0,2009-01-08,08/09,austria,austria,loan,external,19,59132,"aka linz,fussballakademie linz,usv st. ulrich,...",2020/2021


In [31]:
df_th.shape

(129837, 14)

##### Export DataFrame

In [21]:
# Export DataFrame as a CSV file

## Export a copy to the 'archive' subfolder of the TM folder, including the date
df_th.to_csv(data_dir_tm + f'/raw/{short_season_string}/transfer-history/archive/' + f'tm_player_th_all_{short_season_string}_last_updated_{today}.csv', index=None, header=True)

## Export another copy to the TM folder called 'latest' (can be overwritten)
df_th.to_csv(data_dir_tm + f'/raw/{short_season_string}/transfer-history/' + f'tm_player_th_all_{short_season_string}_latest.csv', index=None, header=True)

NameError: name 'df_th' is not defined

#### <a id='#section3.3.6.'>3.3.6. Performance Data</a>

In [12]:
# Run this script to scrape latest version of performance data from TransferMarkt

## Start timer
tic = datetime.datetime.now()

## Create empty list
lst_df_performance = []

## Print time scraping started
print(f'Scraping started at: {tic}')

## Scrape bio information for each player
for player_page in lst_player_urls:
    try:
        dict_output = tm_pull(player_page, performance_data=True, output='pandas')
        df = dict_output['performance_data']
        lst_df_performance.append(df)
        print(f'Performance history data appended for: {player_page}.')
    except:
        pass

## Concatenate DataFrames
df_performance = pd.concat(lst_df_performance)

## Print time scraping ended
print(f'Scraping ended at: {toc}')

## Create attribute for the season    
df_performance['season'] = full_season_string

## End timer
toc = datetime.datetime.now()

## Calculate time take
total_time = (toc-tic).total_seconds()
print(f'Time taken to scrape the performance data of the {len_player_urls:,} players for the {full_season_string} season is: {total_time/60:0.2f} minutes.')

Scraping started at: 2020-12-07 15:54:46.780808
Performance history data appended for: https://www.transfermarkt.com/erik-palmer-brown/profil/spieler/282199.
Performance history data appended for: https://www.transfermarkt.com/manprit-sarkaria/profil/spieler/288690.
Performance history data appended for: https://www.transfermarkt.com/alon-turgeman/profil/spieler/177249.
Performance history data appended for: https://www.transfermarkt.com/bright-edomwonyi/profil/spieler/229857.
Performance history data appended for: https://www.transfermarkt.com/christoph-martschinko/profil/spieler/120191.
Performance history data appended for: https://www.transfermarkt.com/alexander-grunwald/profil/spieler/34763.
Performance history data appended for: https://www.transfermarkt.com/niels-hahn/profil/spieler/404936.
Performance history data appended for: https://www.transfermarkt.com/maudo-jarjue/profil/spieler/450462.
Performance history data appended for: https://www.transfermarkt.com/georg-teigl/profi

Performance history data appended for: https://www.transfermarkt.com/chukwubuike-adamu/profil/spieler/452477.
Performance history data appended for: https://www.transfermarkt.com/luka-sucic/profil/spieler/502722.
Performance history data appended for: https://www.transfermarkt.com/alexander-walke/profil/spieler/1271.
Performance history data appended for: https://www.transfermarkt.com/sekou-koita/profil/spieler/402010.
Performance history data appended for: https://www.transfermarkt.com/youba-diarra/profil/spieler/564451.
Performance history data appended for: https://www.transfermarkt.com/rasmus-kristensen/profil/spieler/369684.
Performance history data appended for: https://www.transfermarkt.com/mohamed-camara/profil/spieler/543382.
Performance history data appended for: https://www.transfermarkt.com/noah-okafor/profil/spieler/346890.
Performance history data appended for: https://www.transfermarkt.com/maximilian-wober/profil/spieler/263361.
Performance history data appended for: htt

Performance history data appended for: https://www.transfermarkt.com/kelvin-arase/profil/spieler/321048.
Performance history data appended for: https://www.transfermarkt.com/lion-schuster/profil/spieler/391560.
Performance history data appended for: https://www.transfermarkt.com/dalibor-velimirovic/profil/spieler/437891.
Performance history data appended for: https://www.transfermarkt.com/deni-alar/profil/spieler/42004.
Performance history data appended for: https://www.transfermarkt.com/lukas-sulzbacher/profil/spieler/375355.
Performance history data appended for: https://www.transfermarkt.com/paul-gobara/profil/spieler/353915.
Performance history data appended for: https://www.transfermarkt.com/dragoljub-savic/profil/spieler/574656.
Performance history data appended for: https://www.transfermarkt.com/taxiarchis-fountas/profil/spieler/192409.
Performance history data appended for: https://www.transfermarkt.com/bernhard-unger/profil/spieler/341761.
Performance history data appended for

Performance history data appended for: https://www.transfermarkt.com/kelvin-yeboah/profil/spieler/589737.
Performance history data appended for: https://www.transfermarkt.com/fabian-koch/profil/spieler/59069.
Performance history data appended for: https://www.transfermarkt.com/ferdinand-oswald/profil/spieler/58360.
Performance history data appended for: https://www.transfermarkt.com/johannes-naschberger/profil/spieler/321874.
Performance history data appended for: https://www.transfermarkt.com/zan-rogelj/profil/spieler/371861.
Performance history data appended for: https://www.transfermarkt.com/nikolai-baden-frederiksen/profil/spieler/400550.
Performance history data appended for: https://www.transfermarkt.com/thanos-petsos/profil/spieler/82110.
Performance history data appended for: https://www.transfermarkt.com/stefan-hager/profil/spieler/158179.
Performance history data appended for: https://www.transfermarkt.com/nemanja-celic/profil/spieler/353899.
Performance history data appended

Performance history data appended for: https://www.transfermarkt.com/thomas-reifeltshammer/profil/spieler/77612.
Performance history data appended for: https://www.transfermarkt.com/kennedy-boateng/profil/spieler/451317.
Performance history data appended for: https://www.transfermarkt.com/samuel-sahin-radlinger/profil/spieler/77563.
Performance history data appended for: https://www.transfermarkt.com/constantin-reiner/profil/spieler/286643.
Performance history data appended for: https://www.transfermarkt.com/felix-seiwald/profil/spieler/394070.
Performance history data appended for: https://www.transfermarkt.com/ante-bajic/profil/spieler/282918.
Performance history data appended for: https://www.transfermarkt.com/lukas-gutlbauer/profil/spieler/371430.
Performance history data appended for: https://www.transfermarkt.com/markus-lackner/profil/spieler/82536.
Performance history data appended for: https://www.transfermarkt.com/seth-paintsil/profil/spieler/372813.
Performance history data a

...

Performance history data appended for: https://www.transfermarkt.com/barbero/profil/spieler/514514.
Performance history data appended for: https://www.transfermarkt.com/oscar-arribas/profil/spieler/476104.
Performance history data appended for: https://www.transfermarkt.com/juanma-bravo/profil/spieler/475833.
Performance history data appended for: https://www.transfermarkt.com/samuel-sosa/profil/spieler/428210.
Performance history data appended for: https://www.transfermarkt.com/ernesto-gomez/profil/spieler/288460.
Performance history data appended for: https://www.transfermarkt.com/aitor-gonzalez/profil/spieler/716157.
Performance history data appended for: https://www.transfermarkt.com/dani-rodriguez/profil/spieler/139435.
Performance history data appended for: https://www.transfermarkt.com/martin-valjent/profil/spieler/277384.
Performance history data appended for: https://www.transfermarkt.com/murilo-de-souza/profil/spieler/345061.
Performance history data appended for: https://www

Performance history data appended for: https://www.transfermarkt.com/santiago-bueno/profil/spieler/438001.
Performance history data appended for: https://www.transfermarkt.com/enric-franquesa/profil/spieler/282700.
Performance history data appended for: https://www.transfermarkt.com/yoel-barcenas/profil/spieler/283473.
Performance history data appended for: https://www.transfermarkt.com/ibrahima-kebe/profil/spieler/785967.
Performance history data appended for: https://www.transfermarkt.com/jonas-ramalho/profil/spieler/69660.
Performance history data appended for: https://www.transfermarkt.com/sebastian-cristoforo/profil/spieler/188157.
Performance history data appended for: https://www.transfermarkt.com/bernardo-espinosa/profil/spieler/64598.
Performance history data appended for: https://www.transfermarkt.com/monchu/profil/spieler/282727.
Performance history data appended for: https://www.transfermarkt.com/juanpe/profil/spieler/115333.
Performance history data appended for: https://w

Performance history data appended for: https://www.transfermarkt.com/manu-barreiro/profil/spieler/72484.
Performance history data appended for: https://www.transfermarkt.com/chris-ramos/profil/spieler/538810.
Performance history data appended for: https://www.transfermarkt.com/carlos-pita/profil/spieler/34566.
Performance history data appended for: https://www.transfermarkt.com/jose-luis-rodriguez/profil/spieler/425028.
Performance history data appended for: https://www.transfermarkt.com/xavi-torres/profil/spieler/65301.
Performance history data appended for: https://www.transfermarkt.com/edu-campabadal/profil/spieler/126722.
Performance history data appended for: https://www.transfermarkt.com/cristian-herrera/profil/spieler/237617.
Performance history data appended for: https://www.transfermarkt.com/roberto-canella/profil/spieler/57785.
Performance history data appended for: https://www.transfermarkt.com/manu-garcia/profil/spieler/277118.
Performance history data appended for: https:/

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Performance history data appended for: https://www.transfermarkt.com/alvaro-raton/profil/spieler/248551.
Performance history data appended for: https://www.transfermarkt.com/jair-amador/profil/spieler/128479.
Performance history data appended for: https://www.transfermarkt.com/ivan-azon/profil/spieler/633306.
Performance history data appended for: https://www.transfermarkt.com/pichu-atienza/profil/spieler/59337.
Performance history data appended for: https://www.transfermarkt.com/alberto-zapater/profil/spieler/26153.
Performance history data appended for: https://www.transfermarkt.com/javi-hernandez/profil/spieler/632893.
Performance history data appended for: https://www.transfermarkt.com/carlos-nieto/profil/spieler/340682.
Performance history data appended for: https://www.transfermarkt.com/cristian-alvarez/profil/spieler/55135.
Performance history data appended for: https://www.transfermarkt.com/gabriel-fernandez/profil/spieler/259930.
Performance history data appended for: https://

In [13]:
# Display DataFrame
df_performance.head()

,season,competition,competition_code,club,in_squad,appearances,ppg,goals,assists,subbed_on,subbed_off,yellow_card,second_yellow_card,red_card,penalties,mins_played,clean_sheets,goals_conceded,player_id,age
0,2020/2021,Bundesliga,A1,Austria Vienna,10,9,1.11,0,0,0,0,1,0,0,0,810,0,0,282199,23
1,2020/2021,ÖFB-Cup,ofb,Austria Vienna,1,1,3.00,0,0,0,0,1,0,0,0,90,0,0,282199,23
2,2020/2021,Bundesliga EL-Play-off,A1EL,Austria Vienna,3,3,1.33,0,0,0,0,0,0,0,0,270,0,0,282199,22
3,2020/2021,Bundesliga,A1,Austria Vienna,23,22,1.59,2,1,3,2,2,0,1,0,1773,0,0,282199,22
4,2020/2021,ÖFB-Cup,ofb,Austria Vienna,1,1,0.00,0,0,0,1,0,0,0,0,68,0,0,282199,22


In [14]:
df_performance.shape

(58581, 20)

##### Export DataFrame

In [15]:
# Export DataFrame as a CSV file

## Export a copy to the 'archive' subfolder of the TM folder, including the date
df_performance.to_csv(data_dir_tm + f'/raw/{short_season_string}/performance/' + f'tm_player_th_all_{short_season_string}_last_updated_{today}.csv', index=None, header=True)

## Export another copy to the TM folder called 'latest' (can be overwritten)
df_performance.to_csv(data_dir_tm + f'/raw/{short_season_string}/performance/archive/' + f'tm_player_th_all_{short_season_string}_latest.csv', index=None, header=True)

#### <a id='#section3.3.7.'>3.3.7. Market Value History</a>

In [16]:
# Run this script to scrape latest version of market value history data from TransferMarkt

## Start timer
tic = datetime.datetime.now()

## Create empty list
lst_df_value_history = []

## Print time scraping started
print(f'Scraping started at: {tic}')

## Scrape bio information for each player
for player_page in lst_player_urls:
    try:
        dict_output = tm_pull(player_page, market_value_history=True, output='pandas')
        df = dict_output['market_value_history']
        lst_df_value_history.append(df)
        print(f'Market value history data appended for: {player_page}.')
    except:
        pass

## Concatenate DataFrames
df_value_history = pd.concat(lst_df_value_history)

## Print time scraping ended
print(f'Scraping ended at: {toc}')

## Create attribute for the season    
df_value_history['season'] = full_season_string

## End timer
toc = datetime.datetime.now()

## Calculate time take
total_time = (toc-tic).total_seconds()
print(f'Time taken to scrape the market value history data of the {len_player_urls:,} players for the {full_season_string} season is: {total_time/60:0.2f} minutes.')

Scraping started at: 2020-12-07 19:40:09.839975
Scraping ended at: 2020-12-07 19:40:08.960476
Time taken to scrape the market value history data of the 21,940 players for the 2020/2021 season is: 0.90 minutes.


In [17]:
# Display DataFrame
df_value_history.head()

,season,competition,competition_code,club,in_squad,appearances,ppg,goals,assists,subbed_on,subbed_off,yellow_card,second_yellow_card,red_card,penalties,mins_played,clean_sheets,goals_conceded,player_id,age
0,2020/2021,Bundesliga,A1,Austria Vienna,10,9,1.11,0,0,0,0,1,0,0,0,810,0,0,282199,23
1,2020/2021,ÖFB-Cup,ofb,Austria Vienna,1,1,3.00,0,0,0,0,1,0,0,0,90,0,0,282199,23
2,2020/2021,Bundesliga EL-Play-off,A1EL,Austria Vienna,3,3,1.33,0,0,0,0,0,0,0,0,270,0,0,282199,22
3,2020/2021,Bundesliga,A1,Austria Vienna,23,22,1.59,2,1,3,2,2,0,1,0,1773,0,0,282199,22
4,2020/2021,ÖFB-Cup,ofb,Austria Vienna,1,1,0.00,0,0,0,1,0,0,0,0,68,0,0,282199,22


In [18]:
df_value_history.shape

(58581, 20)

##### Export DataFrame

In [20]:
# Export DataFrame as a CSV file

## Export a copy to the 'archive' subfolder of the TM folder, including the date
df_value_history.to_csv(data_dir_tm + f'/raw/{short_season_string}/market-value-history/archive/' + f'tm_player_th_all_{short_season_string}_last_updated_{today}.csv', index=None, header=True)

## Export another copy to the TM folder called 'latest' (can be overwritten)
df_value_history.to_csv(data_dir_tm + f'/raw/{short_season_string}/market-value-history/' + f'tm_player_th_all_{short_season_string}_latest.csv', index=None, header=True)

#### <a id='#section3.3.8.'>3.3.8. Join Individual DataFrames</a>

##### Bio and Status data

In [ ]:
# Join the Bio and Status DataFrames to form one, unified DataFrame
df_tm_bio_status = pd.merge(df_bio, df_status, left_on='player_id', right_on='player_id', how='left')

In [ ]:
# Display DataFrame
df_tm_bio_status

In [ ]:
df_tm_bio_status.shape

In [ ]:
# Export DataFrame as a CSV file

## Export a copy to the 'archive' subfolder of the TM folder, including the date
df_tm_bio_status.to_csv(data_dir_tm + f'/raw/{short_season_string}/joined/' + f'tm_bio_status_{short_season_string}_last_updated_{today}.csv', index=None, header=True)

## Export another copy to the TM folder called 'latest' (can be overwritten)
df_tm_bio_status.to_csv(data_dir_tm + f'/raw/{short_season_string}/joined/archive/' + f'tm_bio_status_{short_season_string}_latest.csv', index=None, header=True)

##### Bio, Status, and Transfer History data

In [ ]:
# Join the Bio and Status DataFrames to form one, unified DataFrame
df_tm_bio_status_th = pd.merge(df_tm_bio_status, df_th, left_on='player_id', right_on='player_id', how='left')

In [ ]:
# Display DataFrame
df_tm_bio_status_th

In [ ]:
df_tm_bio_status_th.shape

In [ ]:
# Export DataFrame as a CSV file

## Export a copy to the 'archive' subfolder of the TM folder, including the date
df_tm_bio_status_th.to_csv(data_dir_tm + f'/raw/{short_season_string}/joined/' + f'tm_bio_status_th_{short_season_string}_last_updated_{today}.csv', index=None, header=True)

## Export another copy to the TM folder called 'latest' (can be overwritten)
df_tm_bio_status_th.to_csv(data_dir_tm + f'/raw/{short_season_string}/joined/archive/' + f'tm_bio_status_th_{short_season_string}_latest.csv', index=None, header=True)

##### Bio, Status, and Performance History data

In [ ]:
# Join the Bio and Status DataFrames to form one, unified DataFrame
df_tm_bio_status_performance = pd.merge(df_tm_bio_status, df_performance, left_on='player_id', right_on='player_id', how='left')

In [ ]:
# Display DataFrame
df_tm_bio_status_performance

In [ ]:
df_tm_bio_status_performance.shape

In [ ]:
# Export DataFrame as a CSV file

## Export a copy to the 'archive' subfolder of the TM folder, including the date
df_tm_bio_status_performance.to_csv(data_dir_tm + f'/raw/{short_season_string}/joined/' + f'tm_bio_status_performance_{short_season_string}_last_updated_{today}.csv', index=None, header=True)

## Export another copy to the TM folder called 'latest' (can be overwritten)
df_tm_bio_status_performance.to_csv(data_dir_tm + f'/raw/{short_season_string}/joined/archive/' + f'tm_bio_status_performance_{short_season_string}_latest.csv', index=None, header=True)

##### Bio, Status, and Market Value History data

In [ ]:
# Join the Bio and Status DataFrames to form one, unified DataFrame
df_tm_bio_status_value_history = pd.merge(df_tm_bio_status, df_value_history, left_on='player_id', right_on='player_id', how='left')

In [ ]:
# Display DataFrame
df_tm_bio_status_value_history

In [ ]:
df_tm_bio_status_value_history.shape

In [ ]:
# Export DataFrame as a CSV file

## Export a copy to the 'archive' subfolder of the TM folder, including the date
df_tm_bio_status_value_history.to_csv(data_dir_tm + f'/raw/{short_season_string}/joined/' + f'tm_bio_status_value_history_{short_season_string}_last_updated_{today}.csv', index=None, header=True)

## Export another copy to the TM folder called 'latest' (can be overwritten)
df_tm_bio_status_value_history.to_csv(data_dir_tm + f'/raw/{short_season_string}/joined/archive/' + f'tm_bio_status_value_history_{short_season_string}_latest.csv', index=None, header=True)

## <a id='#section4'>4. Summary</a>
This notebook scrapes data from [TransferMarkt](https://www.transfermarkt.co.uk/) using [Beautifulsoup](https://pypi.org/project/beautifulsoup4/) and the [Tyrone Mings web scraper](https://github.com/FCrSTATS/tyrone_mings) by [FCrSTATS](https://twitter.com/FC_rstats). This landed data is then manipulated as DataFrames using [pandas](http://pandas.pydata.org/).

This data includes Bio and Status data, as well as Transfer History, Performance History and Market Value History data.

## <a id='#section5'>5. Next Steps</a>
The next step is to take this data and engineer it so that it's ready for analysis and to be matched against other data source.

## <a id='#section6'>6. References</a>

#### Data and Web Scraping
*    [tyrone_mings GitHub repository](https://github.com/FCrSTATS/tyrone_mings) by [FCrSTATS](https://github.com/FCrSTATS)
*    [Python Package Index (PyPI) tyrone-mings library](https://pypi.org/project/tyrone-mings/)
*    [Beyond crowd judgments: Data-driven estimation of market value in association football](https://www.sciencedirect.com/science/article/pii/S0377221717304332) by Oliver Müllera, Alexander Simons, and Markus Weinmann.
*    [06/04/2020: BBC - Premier League squads 'drop £1.6bn in value'](https://www.bbc.co.uk/sport/football/52221463).

---

***Visit my website [EddWebster.com](https://www.eddwebster.com) or my [GitHub Repository](https://github.com/eddwebster) for more projects. If you'd like to get in contact, my Twitter handle is [@eddwebster](http://www.twitter.com/eddwebster) and my email is: edd.j.webster@gmail.com.***

[Back to the top](#top)